In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import SimpleITK as sitk

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
sys.path.insert(0, str(ROOT))

import models.grid_proto_fewshot as modulo_alp

DINO_REPO = (
    ROOT / "pretrained_model/hub/facebookresearch_dinov2_main"
)
DINO_PESOS = (
    ROOT / "pretrained_model/hub/checkpoints"
    / "dinov2_vitl14_pretrain.pth"
)
DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"

assert (DINO_REPO / "hubconf.py").is_file()
assert DINO_PESOS.is_file()
assert torch.cuda.is_available()

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False

class FewShotSegDinoLocal(modulo_alp.FewShotSeg):
    def get_encoder(self):
        assert self.config["which_model"] == "dinov2_l14"
        assert self.config["lora"] == 0

        self.encoder = torch.hub.load(
            str(DINO_REPO),
            "dinov2_vitl14",
            source="local",
            pretrained=False,
        )

        pesos = torch.load(
            DINO_PESOS,
            map_location="cpu",
            weights_only=True,
        )

        print(
            "Carregamento DINO:",
            self.encoder.load_state_dict(pesos, strict=True)
        )
        del pesos

        lado = max(
            self.image_size // 14,
            modulo_alp.DEFAULT_FEATURE_SIZE
        )
        self.config["feature_hw"] = [lado, lado]


config_alp = {
    "which_model": "dinov2_l14",
    "cls_name": "grid_proto",
    "proto_grid_size": 8,
    "lora": 0,
    "align": False,
    "debug": False,
    "use_coco_init": False,
    "use_pos_enc": False,
    "use_slice_adapter": False,
    "adapter_layers": 3,
    "dinov2_loss": False,
}

print("Construindo DINOv2-L + ALP...")

modelo_dino = FewShotSegDinoLocal(
    image_size=256,
    pretrained_path=None,
    cfg=config_alp.copy(),
).to(device="cuda", dtype=torch.float32)

modelo_dino.eval().requires_grad_(False)

assert not any(
    p.requires_grad for p in modelo_dino.parameters()
)

print("\n=== CONFIGURAÇÃO DO ENCODER ===")
print("Modelo: DINOv2-L/14")
print("Entrada: 256 × 256")
print("Resolução interna:", (256 // 14) * 14)
print("Grade ALP:", modelo_dino.config["feature_hw"])
print("Encoder congelado: SIM")
print(
    "Parâmetros:",
    sum(p.numel() for p in modelo_dino.encoder.parameters())
)


def carregar_volume(pid):
    imagem = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    ).astype(np.float32)

    mascara = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
    )

    assert imagem.shape == mascara.shape
    assert imagem.shape[1:] == (256, 256)

    media = float(imagem.mean())
    desvio = float(imagem.std())

    assert desvio > 1e-6

    # Igual à normalização usada na avaliação VMamba-MRI.
    return (imagem - media) / desvio, mascara


def preparar(fatia):
    rgb = np.repeat(
        fatia.astype(np.float32)[..., None],
        3,
        axis=2
    )
    return torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1)[None].to("cuda")


# Mesmo suporte usado no experimento VMamba:
# paciente 4, fatia positiva central do rim direito.
classe = 2  # RK

imgs_s, labels_s = carregar_volume(4)
fatias_s = np.flatnonzero(
    np.any(labels_s == classe, axis=(1, 2))
)
z_s = int(fatias_s[len(fatias_s) // 2])

support = preparar(imgs_s[z_s])
fg = torch.from_numpy(
    (labels_s[z_s] == classe).astype(np.float32)
)[None].to("cuda")

# Uma query para diagnóstico: paciente 0,
# fatia positiva central do rim direito.
imgs_q, labels_q = carregar_volume(0)
fatias_q = np.flatnonzero(
    np.any(labels_q == classe, axis=(1, 2))
)
z_q = int(fatias_q[len(fatias_q) // 2])

query = preparar(imgs_q[z_q])
gt = labels_q[z_q] == classe

print("\n=== INFERÊNCIA DIAGNÓSTICA ===")
print(f"Suporte: paciente 4, fatia {z_s}")
print(f"Query: paciente 0, fatia {z_q}")

with torch.inference_mode():
    retorno = modelo_dino(
        [[support]],
        [[fg]],
        [[1 - fg]],
        [query],
        isval=True,
        val_wsize=2,
        show_viz=False
    )

    logits = retorno[0]

    assert logits.shape == (1, 2, 256, 256)
    assert torch.isfinite(logits).all().item()

    pred = (
        logits.argmax(dim=1)[0]
        .cpu().numpy() == 1
    )

tp = int((pred & gt).sum())
fp = int((pred & ~gt).sum())
fn = int((~pred & gt).sum())

dice = 2 * tp / max(2 * tp + fp + fn, 1)
iou = tp / max(tp + fp + fn, 1)

print("Formato logits:", tuple(logits.shape))
print("Pixels ground truth:", int(gt.sum()))
print("Pixels previstos:", int(pred.sum()))
print(f"Dice diagnóstico: {dice:.4f}")
print(f"IoU diagnóstico: {iou:.4f}")
print("\nTESTE DINOv2-L + ALP CONCLUÍDO")

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import json
import ast
import sys
import os

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = ROOT / "exp/baseline_dino_mri_v1"
SAIDA = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"

SCRIPT = EXP / "avaliar_dino_alp_mri.py"
LOG = SAIDA / "console.log"
LANCAMENTO = SAIDA / "lancamento.json"

REF_VMAMBA = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)

assert REF_VMAMBA.is_file()

EXP.mkdir(parents=True, exist_ok=True)
SAIDA.mkdir(parents=True, exist_ok=True)

codigo = r'''
import os
import sys
import csv
import json
import time
import fcntl
import traceback

from pathlib import Path
from functools import lru_cache
from collections import defaultdict

import numpy as np
import torch
import SimpleITK as sitk

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS = ROOT / "data/CHAOST2/chaos_MR_T2_normalized"
SAIDA = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"

DINO_REPO = (
    ROOT / "pretrained_model/hub/facebookresearch_dinov2_main"
)
DINO_PESOS = (
    ROOT / "pretrained_model/hub/checkpoints"
    / "dinov2_vitl14_pretrain.pth"
)

REF_VMAMBA = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)

CSV_FATIAS = SAIDA / "metricas_fatias.csv"
CSV_PACIENTES = SAIDA / "resumo_pacientes.csv"
CSV_GERAL = SAIDA / "resumo_geral.csv"
STATUS = SAIDA / "status.json"

ORGAOS = [(2, "RK"), (3, "LK")]
QUERIES = [0, 1, 2, 3]
SUPORTE = 4
TOTAL = 112

CAMPOS = [
    "encoder", "orgao", "paciente", "fatia",
    "dice", "iou", "precisao", "recall",
    "tp", "fp", "fn", "pixels_gt", "pixels_pred"
]

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import models.grid_proto_fewshot as modulo_alp

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False


def salvar_json(caminho, dados):
    tmp = caminho.with_name(caminho.name + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(tmp, caminho)


class FewShotSegDinoLocal(modulo_alp.FewShotSeg):
    def get_encoder(self):
        assert self.config["which_model"] == "dinov2_l14"
        assert self.config["lora"] == 0

        self.encoder = torch.hub.load(
            str(DINO_REPO),
            "dinov2_vitl14",
            source="local",
            pretrained=False
        )

        pesos = torch.load(
            DINO_PESOS,
            map_location="cpu",
            weights_only=True
        )

        resultado = self.encoder.load_state_dict(
            pesos, strict=True
        )

        print("Pesos DINO:", resultado, flush=True)
        del pesos

        lado = max(
            self.image_size // 14,
            modulo_alp.DEFAULT_FEATURE_SIZE
        )

        self.config["feature_hw"] = [lado, lado]


CONFIG_ALP = {
    "which_model": "dinov2_l14",
    "cls_name": "grid_proto",
    "proto_grid_size": 8,
    "lora": 0,
    "align": False,
    "debug": False,
    "use_coco_init": False,
    "use_pos_enc": False,
    "use_slice_adapter": False,
    "adapter_layers": 3,
    "dinov2_loss": False
}


@lru_cache(maxsize=5)
def volume(pid):
    imagem = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"image_{pid}.nii.gz"))
    ).astype(np.float32)

    labels = sitk.GetArrayFromImage(
        sitk.ReadImage(str(DADOS / f"label_{pid}.nii.gz"))
    )

    assert imagem.shape == labels.shape
    assert imagem.shape[1:] == (256, 256)
    assert np.isfinite(imagem).all()

    media = float(imagem.mean())
    desvio = float(imagem.std())

    assert desvio > 1e-6

    # Exatamente o protocolo de entrada VMamba-MRI.
    imagem = (imagem - media) / desvio

    return imagem, labels


def preparar(fatia):
    rgb = np.repeat(
        fatia.astype(np.float32)[..., None],
        3, axis=2
    )

    tensor = torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1)[None].to(
        "cuda", dtype=torch.float32
    )

    assert tensor.shape == (1, 3, 256, 256)
    return tensor


def chave(registro):
    return (
        registro["orgao"],
        int(registro["paciente"]),
        int(registro["fatia"])
    )


def metricas(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())

    return {
        "dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "iou": tp / max(tp + fp + fn, 1),
        "precisao": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "pixels_gt": int(gt.sum()),
        "pixels_pred": int(pred.sum())
    }


def resumir(registros):
    grupos = defaultdict(list)

    for registro in registros:
        grupos[
            (registro["orgao"], int(registro["paciente"]))
        ].append(registro)

    pacientes = []

    for (orgao, pid), valores in sorted(grupos.items()):
        dice = np.array([
            float(r["dice"]) for r in valores
        ])

        iou = np.array([
            float(r["iou"]) for r in valores
        ])

        pacientes.append({
            "orgao": orgao,
            "paciente": pid,
            "n_fatias": len(valores),
            "dice_medio": float(dice.mean()),
            "dice_dp_fatias": (
                float(dice.std(ddof=1))
                if len(dice) > 1 else 0.0
            ),
            "iou_medio": float(iou.mean())
        })

    with CSV_PACIENTES.open(
        "w", newline="", encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=list(pacientes[0])
        )
        writer.writeheader()
        writer.writerows(pacientes)

    geral = []

    print("\n=== RESULTADO DINOv2-L + ALP ===", flush=True)

    for _, orgao in ORGAOS:
        grupo = [
            r for r in pacientes
            if r["orgao"] == orgao
        ]

        assert len(grupo) == 4

        dice = np.array([
            r["dice_medio"] for r in grupo
        ])

        iou = np.array([
            r["iou_medio"] for r in grupo
        ])

        registro = {
            "encoder": "DINOv2-L",
            "orgao": orgao,
            "n_pacientes": 4,
            "dice_medio": float(dice.mean()),
            "dice_dp_pacientes": float(dice.std(ddof=1)),
            "iou_medio": float(iou.mean())
        }

        geral.append(registro)

        print(
            f"{orgao}: Dice "
            f"{registro['dice_medio']:.4f} ± "
            f"{registro['dice_dp_pacientes']:.4f}",
            flush=True
        )

    with CSV_GERAL.open(
        "w", newline="", encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=list(geral[0])
        )
        writer.writeheader()
        writer.writerows(geral)

    media = np.mean([r["dice_medio"] for r in geral])

    print(
        f"Média dos dois rins: {media:.4f}",
        flush=True
    )


def main():
    assert torch.cuda.is_available()
    assert DINO_PESOS.is_file()
    assert (DINO_REPO / "hubconf.py").is_file()

    SAIDA.mkdir(parents=True, exist_ok=True)

    mapa = json.loads(
        (DADOS / "classmap_1.json").read_text(
            encoding="utf-8"
        )
    )

    # Ordem idêntica à avaliação anterior.
    esperadas = [
        (nome, pid, int(z))
        for _, nome in ORGAOS
        for pid in QUERIES
        for z in mapa[nome][str(pid)]
    ]

    assert len(esperadas) == TOTAL
    assert len(set(esperadas)) == TOTAL

    # Verificar os mesmos casos do VMamba (checkpoint 0).
    referencia = {}

    with REF_VMAMBA.open(
        newline="", encoding="utf-8"
    ) as f:
        for r in csv.DictReader(f):
            if int(r["checkpoint"]) == 0:
                k = chave(r)
                assert k not in referencia
                referencia[k] = r

    assert len(referencia) == TOTAL
    assert set(referencia) == set(esperadas)

    # Permitir retomada sem repetir inferências concluídas.
    anteriores = []

    if CSV_FATIAS.exists():
        assert CSV_FATIAS.read_bytes().endswith(b"\n")

        with CSV_FATIAS.open(
            newline="", encoding="utf-8"
        ) as f:
            reader = csv.DictReader(f)
            assert reader.fieldnames == CAMPOS
            anteriores = list(reader)

        assert len(anteriores) <= TOTAL

        for indice, registro in enumerate(anteriores):
            assert chave(registro) == esperadas[indice], (
                f"Histórico inconsistente na linha {indice+1}"
            )

    concluidos = len(anteriores)

    print(
        f"Retomada: {concluidos}/{TOTAL}",
        flush=True
    )

    if concluidos == TOTAL:
        resumir(anteriores)
        salvar_json(STATUS, {
            "estado": "concluido",
            "concluidos": TOTAL,
            "total": TOTAL
        })
        return

    print("Carregando DINOv2-L...", flush=True)

    modelo = FewShotSegDinoLocal(
        image_size=256,
        pretrained_path=None,
        cfg=CONFIG_ALP.copy()
    ).to("cuda", dtype=torch.float32)

    modelo.eval().requires_grad_(False)

    assert not any(p.requires_grad for p in modelo.parameters())
    assert modelo.config["feature_hw"] == [32, 32]

    # Mesmo suporte one-shot do VMamba.
    imagens_s, labels_s = volume(SUPORTE)
    suportes = {}

    for classe, nome in ORGAOS:
        positivas = np.flatnonzero(
            np.any(labels_s == classe, axis=(1, 2))
        )

        assert len(positivas) > 0

        z = int(positivas[len(positivas) // 2])

        sup_img = preparar(imagens_s[z])
        sup_mask = torch.from_numpy(
            (labels_s[z] == classe).astype(np.float32)
        )[None].to("cuda")

        suportes[classe] = (sup_img, sup_mask)

        print(
            f"Suporte {nome}: paciente {SUPORTE}, fatia {z}",
            flush=True
        )

    if not CSV_FATIAS.exists():
        with CSV_FATIAS.open(
            "x", newline="", encoding="utf-8"
        ) as f:
            csv.DictWriter(
                f, fieldnames=CAMPOS
            ).writeheader()

    inicio = time.perf_counter()

    salvar_json(STATUS, {
        "estado": "executando",
        "concluidos": concluidos,
        "total": TOTAL
    })

    with CSV_FATIAS.open(
        "a", newline="", encoding="utf-8", buffering=1
    ) as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS)

        for nome, pid, z in esperadas[concluidos:]:
            classe = 2 if nome == "RK" else 3
            sup_img, sup_mask = suportes[classe]

            imagens, labels = volume(pid)

            query = preparar(imagens[z])
            gt = labels[z] == classe

            assert gt.any()
            assert int(gt.sum()) == int(
                referencia[(nome, pid, z)]["pixels_gt"]
            )

            with torch.inference_mode():
                retorno = modelo(
                    [[sup_img]],
                    [[sup_mask]],
                    [[1 - sup_mask]],
                    [query],
                    isval=True,
                    val_wsize=2,
                    show_viz=False
                )

                logits = retorno[0]

                assert logits.shape == (1, 2, 256, 256)
                assert torch.isfinite(logits).all().item()

                pred = (
                    logits.argmax(dim=1)[0]
                    .cpu().numpy() == 1
                )

            registro = {
                "encoder": "DINOv2-L",
                "orgao": nome,
                "paciente": pid,
                "fatia": z,
                **metricas(pred, gt)
            }

            writer.writerow(registro)
            concluidos += 1

            if concluidos % 5 == 0 or concluidos == TOTAL:
                salvar_json(STATUS, {
                    "estado": "executando",
                    "concluidos": concluidos,
                    "total": TOTAL,
                    "orgao": nome,
                    "paciente": pid,
                    "tempo_decorrido_s": (
                        time.perf_counter() - inicio
                    )
                })

            if concluidos % 10 == 0:
                print(
                    f"{concluidos}/{TOTAL}",
                    flush=True
                )

            del query, retorno, logits

    with CSV_FATIAS.open(
        newline="", encoding="utf-8"
    ) as f:
        registros = list(csv.DictReader(f))

    assert len(registros) == TOTAL
    assert [chave(r) for r in registros] == esperadas

    resumir(registros)

    salvar_json(STATUS, {
        "estado": "concluido",
        "concluidos": TOTAL,
        "total": TOTAL,
        "tempo_total_s": time.perf_counter() - inicio
    })

    print("AVALIAÇÃO DINO + ALP CONCLUÍDA", flush=True)


if __name__ == "__main__":
    SAIDA.mkdir(parents=True, exist_ok=True)

    with (SAIDA / "execucao.lock").open("w") as lock:
        fcntl.flock(
            lock.fileno(),
            fcntl.LOCK_EX | fcntl.LOCK_NB
        )

        try:
            main()
        except Exception:
            erro = traceback.format_exc()

            salvar_json(STATUS, {
                "estado": "erro",
                "traceback": erro
            })

            print(erro, flush=True)
            raise
'''

ast.parse(codigo, filename=str(SCRIPT))

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "O script existente é diferente. "
        "Nenhum arquivo foi sobrescrito."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

print("=== BASELINE DINOv2-L + ALP — MRI ===")
print("Script:", SCRIPT)
print("Saída:", SAIDA)
print("Resolução: 256 × 256")
print("Suporte: paciente 4")
print("Queries: pacientes 0–3")
print("Total: 112 inferências")
print("Encoder: DINOv2-L congelado")
print("Normalização: z-score por volume")
print(
    "SHA256:",
    hashlib.sha256(codigo.encode()).hexdigest()
)

def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

status = ler_json(SAIDA / "status.json")
lancamento = ler_json(LANCAMENTO)

pid_anterior = lancamento.get("pid")
ativo = False

if pid_anterior:
    try:
        proc = Path(f"/proc/{pid_anterior}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_dino_alp_mri.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

print("\n=== LANÇAMENTO ===")

if status.get("estado") == "concluido":
    print("Avaliação já concluída. Nenhum novo processo.")

elif ativo:
    print("Avaliação já está em execução.")
    print("PID:", pid_anterior)

else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(ROOT),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    registro = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "total": 112,
        "modo": "detached"
    }

    tmp = LANCAMENTO.with_suffix(".json.tmp")
    tmp.write_text(
        json.dumps(registro, indent=2),
        encoding="utf-8"
    )
    os.replace(tmp, LANCAMENTO)

    print("AVALIAÇÃO DINO + ALP LANÇADA")
    print("PID:", processo.pid)
    print("Log:", LOG)
    print("Resultados:", SAIDA)
    print("Execução ainda não verificada.")

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import json
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

DINO = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"

VMAMBA = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_rins_positivas_f0_7ckpts"
)

CHECKPOINTS = 112
ORGAOS = ["RK", "LK"]
PACIENTES = [0, 1, 2, 3]

def ler_json(caminho):
    if not caminho.is_file():
        return {}
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

def ler_csv(caminho):
    if not caminho.is_file():
        return []
    with caminho.open(newline="", encoding="utf-8") as f:
        return [
            r for r in csv.DictReader(f)
            if r.get("orgao") and r.get("dice")
        ]

lancamento = ler_json(DINO / "lancamento.json")
status = ler_json(DINO / "status.json")

pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_dino_alp_mri.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

registros = ler_csv(DINO / "metricas_fatias.csv")
n = len(registros)

print("=" * 68)
print("BASELINE MRI — DINOv2-L CONGELADO + ALP")
print("=" * 68)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))
print(f"Inferências: {n}/112")
print(f"Progresso: {min(n / 112 * 100, 100):.2f}%")

blocos = min(int(n / 112 * 30), 30)
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== PROGRESSO POR ÓRGÃO ===")

contagem = Counter(r["orgao"] for r in registros)

for orgao, esperado in [("RK", 55), ("LK", 57)]:
    print(f"{orgao}: {contagem[orgao]}/{esperado}")

# Conferir correspondência exata dos casos.
def chave(r):
    return (
        r["orgao"],
        int(r["paciente"]),
        int(r["fatia"])
    )

mapa_dino = {chave(r): r for r in registros}

print("\n=== INTEGRIDADE ===")

print(
    "Chaves únicas:",
    len(mapa_dino) == len(registros)
)

referencia = [
    r for r in ler_csv(VMAMBA / "metricas_fatias.csv")
    if int(r["checkpoint"]) == 0
]

mapa_vmamba = {chave(r): r for r in referencia}

print("Casos de referência VMamba:", len(mapa_vmamba))

casos_extras = set(mapa_dino) - set(mapa_vmamba)
print("Casos DINO fora do protocolo:", len(casos_extras))

# Resultados completos somente quando todos os casos
# esperados estiverem presentes.
completo = (
    n == 112
    and len(mapa_dino) == 112
    and len(mapa_vmamba) == 112
    and set(mapa_dino) == set(mapa_vmamba)
)

if completo:
    print("\n=== DICE MÉDIO ± DP ENTRE PACIENTES ===")
    print()
    print(
        f"{'Órgão':<14}"
        f"{'DINOv2-L':>21}"
        f"{'VMamba inicial':>22}"
        f"{'Δ p.p.':>11}"
    )
    print("-" * 68)

    medias_por_orgao = {}

    for orgao in ORGAOS:
        dino_pacientes = []
        vmamba_pacientes = []

        for pid_query in PACIENTES:
            chaves = sorted(
                k for k in mapa_dino
                if k[0] == orgao and k[1] == pid_query
            )

            assert chaves

            dino_pacientes.append(
                np.mean([
                    float(mapa_dino[k]["dice"])
                    for k in chaves
                ])
            )

            vmamba_pacientes.append(
                np.mean([
                    float(mapa_vmamba[k]["dice"])
                    for k in chaves
                ])
            )

        dino_media = float(np.mean(dino_pacientes))
        dino_dp = float(np.std(dino_pacientes, ddof=1))

        vmamba_media = float(np.mean(vmamba_pacientes))
        vmamba_dp = float(np.std(vmamba_pacientes, ddof=1))

        delta = (dino_media - vmamba_media) * 100

        medias_por_orgao[orgao] = (
            dino_media, vmamba_media
        )

        nome = "Rim direito" if orgao == "RK" else "Rim esquerdo"

        print(
            f"{nome:<14}"
            f"{dino_media:8.4f} ± {dino_dp:.4f}"
            f"{vmamba_media:10.4f} ± {vmamba_dp:.4f}"
            f"{delta:+11.2f}"
        )

    dino_geral = np.mean([
        medias_por_orgao[o][0] for o in ORGAOS
    ])
    vmamba_geral = np.mean([
        medias_por_orgao[o][1] for o in ORGAOS
    ])

    print("-" * 68)
    print(
        f"{'Média rins':<14}"
        f"{dino_geral:12.4f}"
        f"{vmamba_geral:21.4f}"
        f"{(dino_geral - vmamba_geral)*100:+11.2f}"
    )

    print("\n=== PREVISÕES SEM FOREGROUND ===")

    for modelo_nome, mapa in [
        ("DINOv2-L", mapa_dino),
        ("VMamba-Tiny inicial", mapa_vmamba)
    ]:
        vazias = sum(
            int(r["pixels_pred"]) == 0
            for r in mapa.values()
        )
        print(f"{modelo_nome}: {vazias}/112")

else:
    print("\n=== RESULTADOS ===")
    print("Aguardando a conclusão das 112 inferências.")

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "Sem detalhes."))

elif status.get("estado") == "concluido" and completo:
    print("AVALIAÇÃO DINOv2-L + ALP CONCLUÍDA")

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo ou conclusão ainda não confirmada.")

print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

log = DINO / "console.log"

if log.is_file():
    with log.open(
        encoding="utf-8", errors="replace"
    ) as f:
        for linha in deque(f, maxlen=18):
            print(linha.rstrip())
else:
    print("Log ainda não disponível.")

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import json
import ast
import csv
import sys
import os

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXP = ROOT / "exp/baseline_dino_mri_v1"

ORIGINAL = (
    ROOT / "exp/eft_mri_chaos_v1/avaliar_alp_sam_mri.py"
)

REF_ALP = (
    ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"
    / "metricas_fatias.csv"
)

SAIDA = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp_sam"
SCRIPT = EXP / "avaliar_dino_alp_sam_mri.py"
LOG = SAIDA / "console.log"
LANCAMENTO = SAIDA / "lancamento.json"

assert ORIGINAL.is_file(), ORIGINAL
assert REF_ALP.is_file(), REF_ALP
assert (
    ROOT / "pretrained_model/hub/checkpoints/"
    "dinov2_vitl14_pretrain.pth"
).is_file()
assert (ROOT / "pretrained_model/sam_vit_h.pth").is_file()

with REF_ALP.open(newline="", encoding="utf-8") as f:
    referencias = list(csv.DictReader(f))

assert len(referencias) == 112
assert len({
    (r["orgao"], int(r["paciente"]), int(r["fatia"]))
    for r in referencias
}) == 112

EXP.mkdir(parents=True, exist_ok=True)
SAIDA.mkdir(parents=True, exist_ok=True)

codigo = ORIGINAL.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    quantidade = codigo.count(antigo)
    assert quantidade == 1, (
        f"Trecho não encontrado ou ambíguo "
        f"({quantidade} ocorrências): {antigo[:90]!r}"
    )
    codigo = codigo.replace(antigo, novo, 1)

# 1. Caminhos de resultados e referência ALP.
trocar(
    'TREINO = ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"',
    'TREINO = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"'
)

trocar(
    'SAIDA = TREINO / "avaliacao_alp_sam_rins_f0_7ckpts"',
    'SAIDA = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp_sam"'
)

trocar(
    '''REF_ALP = (
    TREINO / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)''',
    'REF_ALP = TREINO / "metricas_fatias.csv"'
)

# 2. Apenas o baseline congelado.
trocar(
    'CHECKPOINTS = [0, 200, 500, 1000, 2000, 5000, 10000]',
    'CHECKPOINTS = [0]'
)

trocar('TOTAL = 784', 'TOTAL = 112')

# 3. O CSV DINO não tem coluna "checkpoint".
# Internamente atribuímos passo 0 ao baseline congelado.
trocar(
    'int(registro["checkpoint"]),\n        registro["orgao"],',
    'int(registro.get("checkpoint", 0)),\n        registro["orgao"],'
)

# 4. Remover a função de checkpoints VMamba.
inicio = codigo.index("def checkpoint(passo):")
fim = codigo.index("@lru_cache(maxsize=5)", inicio)

codigo = codigo[:inicio] + codigo[fim:]

# 5. Substituir somente a construção do encoder.
construtor = '''
import models.grid_proto_fewshot as modulo_alp

DINO_REPO = (
    ROOT / "pretrained_model/hub/facebookresearch_dinov2_main"
)
DINO_PESOS = (
    ROOT / "pretrained_model/hub/checkpoints"
    / "dinov2_vitl14_pretrain.pth"
)

class FewShotSegDinoLocal(modulo_alp.FewShotSeg):
    def get_encoder(self):
        assert self.config["which_model"] == "dinov2_l14"
        assert self.config["lora"] == 0

        self.encoder = torch.hub.load(
            str(DINO_REPO),
            "dinov2_vitl14",
            source="local",
            pretrained=False
        )

        pesos = torch.load(
            DINO_PESOS,
            map_location="cpu",
            weights_only=True
        )

        resultado = self.encoder.load_state_dict(
            pesos, strict=True
        )
        print("Pesos DINO:", resultado, flush=True)
        del pesos

        lado = max(
            self.image_size // 14,
            modulo_alp.DEFAULT_FEATURE_SIZE
        )
        self.config["feature_hw"] = [lado, lado]


CONFIG_ALP = {
    "which_model": "dinov2_l14",
    "cls_name": "grid_proto",
    "proto_grid_size": 8,
    "lora": 0,
    "align": False,
    "debug": False,
    "use_coco_init": False,
    "use_pos_enc": False,
    "use_slice_adapter": False,
    "adapter_layers": 3,
    "dinov2_loss": False
}
'''

trocar(
    'from eft_modelo_672 import construir_modelo',
    construtor
)

trocar(
    '''    for passo in CHECKPOINTS:
        assert checkpoint(passo).is_file()''',
    '''    assert DINO_PESOS.is_file()
    assert (DINO_REPO / "hubconf.py").is_file()'''
)

trocar(
    '''    modelo = construir_modelo("tiny", repo=ROOT)
    modelo.cuda().eval().requires_grad_(False)''',
    '''    modelo = FewShotSegDinoLocal(
        image_size=256,
        pretrained_path=None,
        cfg=CONFIG_ALP.copy()
    ).to("cuda", dtype=torch.float32)

    modelo.eval().requires_grad_(False)

    assert not any(
        p.requires_grad for p in modelo.parameters()
    )
    assert modelo.config["feature_hw"] == [32, 32]'''
)

# 6. Não carregar checkpoints EFT durante a avaliação.
inicio = codigo.index(
    "            if passo != checkpoint_atual:"
)
fim = codigo.index(
    '            classe = 2 if nome == "RK" else 3',
    inicio
)

codigo = (
    codigo[:inicio]
    + '''            if passo != checkpoint_atual:
                modelo.eval()
                protosam.eval()
                checkpoint_atual = passo
                print(
                    "\\nBASELINE DINOv2-L CONGELADO",
                    flush=True
                )

'''
    + codigo[fim:]
)

# 7. Garantias antes do lançamento.
ast.parse(codigo, filename=str(SCRIPT))

assert 'CHECKPOINTS = [0]' in codigo
assert 'TOTAL = 112' in codigo
assert 'SUPORTE = 4' in codigo
assert 'QUERIES = [0, 1, 2, 3]' in codigo
assert 'ORGAOS = [(2, "RK"), (3, "LK")]' in codigo
assert 'class FewShotSegDinoLocal' in codigo
assert 'image_size=256' in codigo
assert 'modelo.eval().requires_grad_(False)' in codigo
assert 'cfg["image_size"] = [1024, 1024]' in codigo
assert 'ProtoSAM(' in codigo
assert 'ALPNetWrapper(modelo)' in codigo
assert 'dice_alp' in codigo
assert 'dice_sam' in codigo
assert 'sam_executado' in codigo
assert 'checkpoint(passo)' not in codigo
assert 'construir_modelo("tiny"' not in codigo

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "O script existente difere da versão esperada."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

sha_script = hashlib.sha256(
    codigo.encode("utf-8")
).hexdigest()

config = {
    "dataset": "CHAOST2",
    "modalidade": "MRI T2-SPIR",
    "encoder": "DINOv2-L/14 congelado",
    "resolucao_encoder": 256,
    "sam": "SAM-H",
    "resolucao_sam": 1024,
    "normalizacao": "z-score por volume",
    "suporte": 4,
    "queries": [0, 1, 2, 3],
    "classes": ["RK", "LK"],
    "total_inferencias": 112,
    "script_sha256": sha_script,
    "referencia_alp_sha256": hashlib.sha256(
        REF_ALP.read_bytes()
    ).hexdigest()
}

CONFIG = SAIDA / "config_avaliacao.json"

if CONFIG.exists():
    assert json.loads(
        CONFIG.read_text(encoding="utf-8")
    ) == config
else:
    tmp = CONFIG.with_name(CONFIG.name + ".tmp")
    tmp.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(tmp, CONFIG)

print("=" * 62)
print("BASELINE DINOv2-L + ALP + SAM-H — CHAOS MRI")
print("=" * 62)
print("Script:", SCRIPT)
print("SHA256:", sha_script)
print("Suporte: paciente 4")
print("Queries: pacientes 0–3")
print("Rim direito: 55 fatias")
print("Rim esquerdo: 57 fatias")
print("Total: 112 inferências")
print("Saída:", SAIDA)

def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

status = ler_json(SAIDA / "status.json")
lancamento = ler_json(LANCAMENTO)

pid_anterior = lancamento.get("pid")
ativo = False

if pid_anterior:
    try:
        proc = Path(f"/proc/{pid_anterior}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_dino_alp_sam_mri.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

print("\n=== LANÇAMENTO ===")

if status.get("estado") == "concluido":
    print("Avaliação já concluída.")

elif ativo:
    print("Avaliação em execução. PID:", pid_anterior)

else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(ROOT),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    registro = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "total": 112,
        "modo": "detached"
    }

    tmp = LANCAMENTO.with_name(LANCAMENTO.name + ".tmp")
    tmp.write_text(
        json.dumps(registro, indent=2),
        encoding="utf-8"
    )
    os.replace(tmp, LANCAMENTO)

    print("AVALIAÇÃO DINO + ALP + SAM LANÇADA")
    print("PID:", processo.pid)
    print("Log:", LOG)
    print("Resultados:", SAIDA)
    print("Execução solicitada; ainda não verificada.")

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import json
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

DINO = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp_sam"
DINO_ALP = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"

VMAMBA = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_sam_rins_f0_7ckpts"
)

TOTAL = 112
ORGAOS = ["RK", "LK"]
PACIENTES = [0, 1, 2, 3]


def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}


def ler_csv(path):
    if not path.is_file():
        return []

    registros = []
    with path.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if not r.get("orgao") or not r.get("paciente"):
                continue
            registros.append(r)
    return registros


def chave(r):
    return (
        r["orgao"],
        int(r["paciente"]),
        int(r["fatia"])
    )


def media_pacientes(mapa, orgao, coluna):
    medias = []

    for pid in PACIENTES:
        valores = [
            float(r[coluna])
            for k, r in mapa.items()
            if k[0] == orgao and k[1] == pid
        ]
        assert valores, (orgao, pid)
        medias.append(np.mean(valores))

    return float(np.mean(medias)), float(
        np.std(medias, ddof=1)
    )


lancamento = ler_json(DINO / "lancamento.json")
status = ler_json(DINO / "status.json")

pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_dino_alp_sam_mri.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

registros = ler_csv(DINO / "metricas_fatias.csv")
n = len(registros)

print("=" * 70)
print("CHAOS MRI — DINOv2-L + ALP + SAM-H")
print("=" * 70)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))
print(f"Inferências: {n}/{TOTAL}")
print(f"Progresso: {min(100, n/TOTAL*100):.2f}%")

blocos = min(30, int(n / TOTAL * 30))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

contagem = Counter(r["orgao"] for r in registros)

print("\n=== PROGRESSO POR ÓRGÃO ===")
print(f"Rim direito: {contagem['RK']}/55")
print(f"Rim esquerdo: {contagem['LK']}/57")

# Conferir as mesmas fatias do DINO ALP.
referencia = ler_csv(DINO_ALP / "metricas_fatias.csv")

mapa_ref = {chave(r): r for r in referencia}
mapa_dino = {chave(r): r for r in registros}

integridade = (
    len(mapa_dino) == n
    and len(mapa_ref) == TOTAL
    and set(mapa_dino).issubset(set(mapa_ref))
)

completo = (
    n == TOTAL
    and integridade
    and set(mapa_dino) == set(mapa_ref)
)

print("\n=== INTEGRIDADE ===")
print("Chaves únicas e válidas:", integridade)
print("Todos os casos presentes:", completo)

if completo:
    for k, r in mapa_dino.items():
        assert np.isclose(
            float(r["dice_alp"]),
            float(mapa_ref[k]["dice"]),
            atol=1e-8
        ), f"Divergência ALP: {k}"

    print("Dice ALP coincide com baseline anterior: SIM")

    print("\n=== DINOv2-L: ALP versus ALP + SAM ===")
    print("Média ± DP entre os quatro pacientes")
    print()

    print(
        f"{'Órgão':<15}"
        f"{'ALP':>18}"
        f"{'ALP + SAM':>20}"
        f"{'Δ p.p.':>10}"
    )
    print("-" * 67)

    resultados = {}

    for orgao in ORGAOS:
        alp, dp_alp = media_pacientes(
            mapa_dino, orgao, "dice_alp"
        )
        sam, dp_sam = media_pacientes(
            mapa_dino, orgao, "dice_sam"
        )

        resultados[orgao] = {"alp": alp, "sam": sam}

        nome = "Rim direito" if orgao == "RK" else "Rim esquerdo"

        print(
            f"{nome:<15}"
            f"{alp:8.4f} ± {dp_alp:.4f}"
            f"{sam:10.4f} ± {dp_sam:.4f}"
            f"{(sam-alp)*100:+10.2f}"
        )

    alp_geral = np.mean([
        resultados[o]["alp"] for o in ORGAOS
    ])
    sam_geral = np.mean([
        resultados[o]["sam"] for o in ORGAOS
    ])

    print("-" * 67)
    print(
        f"{'Média dos rins':<15}"
        f"{alp_geral:12.4f}"
        f"{sam_geral:20.4f}"
        f"{(sam_geral-alp_geral)*100:+10.2f}"
    )

    print("\n=== COMPARAÇÃO COM VMAMBA-TINY + SAM-H ===")

    vmamba = ler_csv(VMAMBA / "metricas_fatias.csv")

    comparacoes = [
        ("VMamba inicial", 0),
        ("VMamba EFT 5000", 5000),
        ("VMamba EFT 10000", 10000)
    ]

    print(f"{'Modelo':<25}{'Dice médio':>13}{'Δ DINO p.p.':>15}")
    print("-" * 55)

    print(f"{'DINOv2-L congelado':<25}{sam_geral:13.4f}{0:15.2f}")

    for nome, passo in comparacoes:
        subconjunto = [
            r for r in vmamba
            if int(r["checkpoint"]) == passo
        ]

        mapa_v = {chave(r): r for r in subconjunto}

        assert len(mapa_v) == TOTAL
        assert set(mapa_v) == set(mapa_dino)

        medias = [
            media_pacientes(
                mapa_v, orgao, "dice_sam"
            )[0]
            for orgao in ORGAOS
        ]

        media_v = float(np.mean(medias))

        print(
            f"{nome:<25}"
            f"{media_v:13.4f}"
            f"{(sam_geral-media_v)*100:+15.2f}"
        )

    sem_sam = sum(
        int(r["sam_executado"]) == 0
        for r in registros
    )

    print("\n=== EXECUÇÃO DO SAM ===")
    print(f"Sem prompts: {sem_sam}/{TOTAL}")
    print(f"SAM executado: {TOTAL-sem_sam}/{TOTAL}")

else:
    print("\nResultados completos ainda não disponíveis.")

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "Sem detalhes."))

elif status.get("estado") == "concluido" and completo:
    print("AVALIAÇÃO DINOv2-L + ALP + SAM-H CONCLUÍDA")

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

log = DINO / "console.log"

if log.is_file():
    with log.open(
        encoding="utf-8", errors="replace"
    ) as f:
        for linha in deque(f, maxlen=18):
            print(linha.rstrip())
else:
    print("Log ainda não disponível.")

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import os
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

DINO_DIR = (
    ROOT / "reproducao/baseline_dino_mri_256_fold0_alp_sam"
)

VMAMBA_DIR = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_sam_rins_f0_7ckpts"
)

CHECKPOINTS = [0, 5000, 10000]
ORGAOS = ["RK", "LK"]
PACIENTES = [0, 1, 2, 3]

def carregar_csv(caminho):
    assert caminho.is_file(), caminho
    with caminho.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def chave(r):
    return (
        r["orgao"],
        int(r["paciente"]),
        int(r["fatia"])
    )

dino = carregar_csv(DINO_DIR / "metricas_fatias.csv")
vmamba = carregar_csv(VMAMBA_DIR / "metricas_fatias.csv")

assert len(dino) == 112

mapa_dino = {chave(r): r for r in dino}
assert len(mapa_dino) == 112

registros = []

print("=" * 65)
print("COMPARAÇÃO PAREADA — DINOv2-L versus VMAMBA-TINY")
print("=" * 65)
print("Métrica: Dice após ALP + SAM-H")
print("Unidade independente: paciente")
print("Resolução: 256 × 256")

for passo in CHECKPOINTS:
    subconjunto = [
        r for r in vmamba
        if int(r["checkpoint"]) == passo
    ]

    mapa_vmamba = {
        chave(r): r for r in subconjunto
    }

    assert len(subconjunto) == 112
    assert len(mapa_vmamba) == 112
    assert set(mapa_dino) == set(mapa_vmamba)

    print(f"\n=== DINO versus VMAMBA — passo {passo} ===")
    print("Diferença positiva = vantagem do DINO")
    print()
    print(
        f"{'Paciente':>9} | "
        f"{'Δ direito':>11} | "
        f"{'Δ esquerdo':>12} | "
        f"{'Δ médio':>10}"
    )
    print("-" * 54)

    diferencas_pacientes = []

    for pid in PACIENTES:
        diferencas_orgaos = []

        for orgao in ORGAOS:
            chaves = sorted(
                k for k in mapa_dino
                if k[0] == orgao and k[1] == pid
            )

            assert chaves

            dino_dice = float(np.mean([
                float(mapa_dino[k]["dice_sam"])
                for k in chaves
            ]))

            vmamba_dice = float(np.mean([
                float(mapa_vmamba[k]["dice_sam"])
                for k in chaves
            ]))

            delta = dino_dice - vmamba_dice

            diferencas_orgaos.append(delta)

            registros.append({
                "checkpoint_vmamba": passo,
                "paciente": pid,
                "orgao": orgao,
                "n_fatias": len(chaves),
                "dice_dino": dino_dice,
                "dice_vmamba": vmamba_dice,
                "delta_dice": delta
            })

        media_paciente = float(np.mean(diferencas_orgaos))
        diferencas_pacientes.append(media_paciente)

        print(
            f"{pid:9d} | "
            f"{diferencas_orgaos[0]*100:+10.2f} | "
            f"{diferencas_orgaos[1]*100:+11.2f} | "
            f"{media_paciente*100:+9.2f}"
        )

    media = float(np.mean(diferencas_pacientes))
    dp = float(np.std(diferencas_pacientes, ddof=1))

    positivos = sum(x > 0 for x in diferencas_pacientes)

    print("-" * 54)
    print(f"Vantagem média DINO: {media*100:+.2f} p.p.")
    print(f"DP das diferenças entre pacientes: {dp*100:.2f} p.p.")
    print(f"Pacientes em que DINO foi superior: {positivos}/4")

# Verificar e salvar resultados.
SAIDA = DINO_DIR / "comparacao_pareada_vmamba.csv"

campos = [
    "checkpoint_vmamba",
    "paciente",
    "orgao",
    "n_fatias",
    "dice_dino",
    "dice_vmamba",
    "delta_dice"
]

assert len(registros) == 3 * 4 * 2

temporario = SAIDA.with_name(SAIDA.name + ".tmp")

with temporario.open(
    "w", newline="", encoding="utf-8"
) as f:
    escritor = csv.DictWriter(f, fieldnames=campos)
    escritor.writeheader()
    escritor.writerows(registros)

if SAIDA.exists():
    assert SAIDA.read_bytes() == temporario.read_bytes(), (
        "Já existe um resultado diferente neste caminho."
    )
    temporario.unlink()
    print("\nArquivo anterior conferido, sem alterações.")
else:
    os.replace(temporario, SAIDA)
    print("\nResultados salvos.")

print("CSV:", SAIDA)
print("Total de comparações por paciente e órgão:", len(registros))
print("\nCOMPARAÇÃO CONCLUÍDA")

In [ ]:
from pathlib import Path
import importlib.util
import csv

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = (
    ROOT / "exp/baseline_dino_mri_v1"
    / "avaliar_dino_alp_mri.py"
)

REF_256 = (
    ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"
    / "metricas_fatias.csv"
)

assert SCRIPT.is_file()
assert REF_256.is_file()
assert torch.cuda.is_available()

# Reutilizar as classes e a normalização do avaliador validado.
spec = importlib.util.spec_from_file_location(
    "avaliador_dino_referencia", SCRIPT
)
modulo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(modulo)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False

print("=== CARREGAMENTO DINOv2-L — 672 ===")

modelo_672 = modulo.FewShotSegDinoLocal(
    image_size=672,
    pretrained_path=None,
    cfg=modulo.CONFIG_ALP.copy()
).to("cuda", dtype=torch.float32)

modelo_672.eval().requires_grad_(False)

assert modelo_672.config["feature_hw"] == [48, 48]
assert not any(
    p.requires_grad for p in modelo_672.parameters()
)

print("Grade ALP:", modelo_672.config["feature_hw"])
print("Encoder congelado: SIM")

# Mesmo par usado no diagnóstico inicial.
classe = 2  # rim direito

imgs_s, labels_s = modulo.volume(4)
imgs_q, labels_q = modulo.volume(0)

fatias_s = np.flatnonzero(
    np.any(labels_s == classe, axis=(1, 2))
)
fatias_q = np.flatnonzero(
    np.any(labels_q == classe, axis=(1, 2))
)

z_s = int(fatias_s[len(fatias_s) // 2])
z_q = int(fatias_q[len(fatias_q) // 2])

support_256 = modulo.preparar(imgs_s[z_s])
query_256 = modulo.preparar(imgs_q[z_q])

fg_256 = torch.from_numpy(
    (labels_s[z_s] == classe).astype(np.float32)
)[None, None].to("cuda")

# Imagens: interpolação bilinear.
# Máscaras: interpolação nearest, preservando rótulos.
support_672 = F.interpolate(
    support_256,
    size=(672, 672),
    mode="bilinear",
    align_corners=False
)

query_672 = F.interpolate(
    query_256,
    size=(672, 672),
    mode="bilinear",
    align_corners=False
)

fg_672 = F.interpolate(
    fg_256,
    size=(672, 672),
    mode="nearest"
)[:, 0]

assert support_672.shape == (1, 3, 672, 672)
assert query_672.shape == (1, 3, 672, 672)
assert fg_672.shape == (1, 672, 672)
assert set(torch.unique(fg_672).tolist()) <= {0.0, 1.0}

print("\n=== INFERÊNCIA DIAGNÓSTICA ===")
print(f"Suporte: paciente 4, fatia {z_s}")
print(f"Query: paciente 0, fatia {z_q}")

with torch.inference_mode():
    retorno = modelo_672(
        [[support_672]],
        [[fg_672]],
        [[1 - fg_672]],
        [query_672],
        isval=True,
        val_wsize=2,
        show_viz=False
    )

    logits_672 = retorno[0]

    print(
        "Logits 672:",
        tuple(logits_672.shape)
    )

    assert logits_672.shape == (1, 2, 672, 672)
    assert torch.isfinite(logits_672).all().item()

    # Retornar logits à resolução original antes do argmax.
    logits_256 = F.interpolate(
        logits_672,
        size=(256, 256),
        mode="bilinear",
        align_corners=False
    )

    pred = (
        logits_256.argmax(dim=1)[0]
        .cpu().numpy() == 1
    )

gt = labels_q[z_q] == classe

tp = int((pred & gt).sum())
fp = int((pred & ~gt).sum())
fn = int((~pred & gt).sum())

dice_672 = 2 * tp / max(2 * tp + fp + fn, 1)
iou_672 = tp / max(tp + fp + fn, 1)

# Recuperar resultado do mesmo caso em 256.
with REF_256.open(newline="", encoding="utf-8") as f:
    candidatos = [
        r for r in csv.DictReader(f)
        if r["orgao"] == "RK"
        and int(r["paciente"]) == 0
        and int(r["fatia"]) == z_q
    ]

assert len(candidatos) == 1

dice_256 = float(candidatos[0]["dice"])

print("\n=== COMPARAÇÃO DO MESMO CASO ===")
print(f"Dice DINO 256: {dice_256:.4f}")
print(f"Dice DINO 672: {dice_672:.4f}")
print(f"IoU DINO 672: {iou_672:.4f}")
print(f"Δ Dice: {(dice_672-dice_256)*100:+.2f} p.p.")
print("Pixels GT:", int(gt.sum()))
print("Pixels previstos em 672, avaliados a 256:", int(pred.sum()))

print("\nTESTE DIAGNÓSTICO CONCLUÍDO")

# Liberar o segundo modelo da GPU.
del modelo_672
del retorno, logits_672, logits_256
del support_672, query_672, fg_672
del support_256, query_256, fg_256
torch.cuda.empty_cache()

In [ ]:
from pathlib import Path
import ast
import hashlib
import json
import os
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ORIGINAL = (
    ROOT / "exp/baseline_dino_mri_v1/"
    "avaliar_dino_alp_mri.py"
)

SCRIPT = (
    ROOT / "exp/baseline_dino_mri_v1/"
    "avaliar_dino_alp_mri_672.py"
)

SAIDA = (
    ROOT / "reproducao/"
    "baseline_dino_mri_672_fold0_alp"
)

LOG = SAIDA / "console.log"
LANCAMENTO = SAIDA / "lancamento.json"
STATUS = SAIDA / "status.json"
CONFIG = SAIDA / "config_avaliacao.json"

assert ORIGINAL.is_file()
SAIDA.mkdir(parents=True, exist_ok=True)

codigo = ORIGINAL.read_text(encoding="utf-8")

# Conferir que estamos adaptando exatamente o
# avaliador de 256 já utilizado anteriormente.
sha_origem = hashlib.sha256(
    codigo.encode("utf-8")
).hexdigest()

assert sha_origem == (
    "d6c7ca2aca9d116c13ede26f6e824"
    "bda04e073d6d5cfa5326b602066f4d16ae6"
), "O avaliador original foi alterado."

def trocar(antigo, novo):
    global codigo

    quantidade = codigo.count(antigo)

    assert quantidade == 1, (
        f"Substituição ambígua: {quantidade} "
        f"ocorrências de {antigo[:80]!r}"
    )

    codigo = codigo.replace(antigo, novo, 1)


# 1. Novo diretório de resultados
trocar(
    'SAIDA = ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"',
    'SAIDA = ROOT / "reproducao/baseline_dino_mri_672_fold0_alp"'
)

# 2. Interpolação
trocar(
    "import torch\nimport SimpleITK as sitk",
    "import torch\nimport torch.nn.functional as F\n"
    "import SimpleITK as sitk"
)

# 3. Imagens 256 -> 672
trocar(
    '''    assert tensor.shape == (1, 3, 256, 256)
    return tensor''',
    '''    assert tensor.shape == (1, 3, 256, 256)

    return F.interpolate(
        tensor,
        size=(672, 672),
        mode="bilinear",
        align_corners=False
    )'''
)

# 4. Construir encoder DINO com resolução 672
trocar(
    '''        image_size=256,
        pretrained_path=None,''',
    '''        image_size=672,
        pretrained_path=None,'''
)

trocar(
    'assert modelo.config["feature_hw"] == [32, 32]',
    'assert modelo.config["feature_hw"] == [48, 48]'
)

# 5. Redimensionar máscara de suporte sem
# introduzir valores intermediários entre 0 e 1.
trocar(
    '''        sup_mask = torch.from_numpy(
            (labels_s[z] == classe).astype(np.float32)
        )[None].to("cuda")

        suportes[classe] = (sup_img, sup_mask)''',
    '''        sup_mask_256 = torch.from_numpy(
            (labels_s[z] == classe).astype(np.float32)
        )[None, None].to("cuda")

        sup_mask = F.interpolate(
            sup_mask_256,
            size=(672, 672),
            mode="nearest"
        )[:, 0]

        assert sup_mask.shape == (1, 672, 672)
        assert torch.all(
            (sup_mask == 0) | (sup_mask == 1)
        ).item()

        suportes[classe] = (sup_img, sup_mask)'''
)

# 6. Retornar os logits a 256 ANTES do argmax.
# Ground truth permanece intocada, em 256.
trocar(
    '''                assert logits.shape == (1, 2, 256, 256)
                assert torch.isfinite(logits).all().item()

                pred = (
                    logits.argmax(dim=1)[0]
                    .cpu().numpy() == 1
                )''',
    '''                assert logits.shape == (1, 2, 672, 672)
                assert torch.isfinite(logits).all().item()

                logits_256 = F.interpolate(
                    logits,
                    size=(256, 256),
                    mode="bilinear",
                    align_corners=False
                )

                pred = (
                    logits_256.argmax(dim=1)[0]
                    .cpu().numpy() == 1
                )'''
)

# 7. Verificações
ast.parse(codigo, filename=str(SCRIPT))

assert 'image_size=672' in codigo
assert 'size=(672, 672)' in codigo
assert 'mode="nearest"' in codigo
assert 'assert logits.shape == (1, 2, 672, 672)' in codigo
assert 'assert modelo.config["feature_hw"] == [48, 48]' in codigo
assert 'gt = labels[z] == classe' in codigo
assert 'TOTAL = 112' in codigo
assert 'SUPORTE = 4' in codigo
assert 'QUERIES = [0, 1, 2, 3]' in codigo

if SCRIPT.exists():
    assert SCRIPT.read_text(encoding="utf-8") == codigo, (
        "Existe um script diferente nesse caminho."
    )
else:
    SCRIPT.write_text(codigo, encoding="utf-8")

sha_script = hashlib.sha256(
    codigo.encode("utf-8")
).hexdigest()

config = {
    "encoder": "DINOv2-L/14 congelado",
    "modalidade": "MRI CHAOS T2-SPIR",
    "entrada_original": 256,
    "entrada_encoder": 672,
    "grade_alp": [48, 48],
    "kernel_alp": [6, 6],
    "interpolacao_imagem": "bilinear align_corners=False",
    "interpolacao_mascara": "nearest",
    "interpolacao_logits": "bilinear align_corners=False",
    "resolucao_metricas": 256,
    "normalizacao": "z-score por volume",
    "suporte": 4,
    "queries": [0, 1, 2, 3],
    "orgaos": ["RK", "LK"],
    "inferencias": 112,
    "script_sha256": sha_script,
    "origem_sha256": sha_origem
}

if CONFIG.exists():
    assert json.loads(
        CONFIG.read_text(encoding="utf-8")
    ) == config
else:
    tmp = CONFIG.with_name(CONFIG.name + ".tmp")
    tmp.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(tmp, CONFIG)

print("=" * 66)
print("DINOv2-L + ALP — MRI, ENTRADA 672")
print("=" * 66)
print("Script:", SCRIPT)
print("SHA256:", sha_script)
print("Resolução original: 256 × 256")
print("Resolução encoder: 672 × 672")
print("Grade ALP: 48 × 48")
print("Kernel ALP: 6 × 6")
print("Suporte: paciente 4")
print("Queries: pacientes 0–3")
print("Total de inferências: 112")
print("Diretório:", SAIDA)

# 8. Lançamento independente do Jupyter
def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

status = ler_json(STATUS)
lancamento = ler_json(LANCAMENTO)

pid_anterior = lancamento.get("pid")
ativo = False

if pid_anterior:
    try:
        proc = Path(f"/proc/{pid_anterior}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_dino_alp_mri_672.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

print("\n=== LANÇAMENTO ===")

if status.get("estado") == "concluido":
    print("Avaliação já concluída. Nenhum processo iniciado.")

elif ativo:
    print("Avaliação já em execução.")
    print("PID:", pid_anterior)

else:
    with LOG.open("a", encoding="utf-8") as log:
        processo = subprocess.Popen(
            [sys.executable, "-u", str(SCRIPT)],
            cwd=str(ROOT),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    registro = {
        "pid": processo.pid,
        "script": str(SCRIPT),
        "log": str(LOG),
        "total": 112,
        "modo": "detached"
    }

    tmp = LANCAMENTO.with_name(LANCAMENTO.name + ".tmp")
    tmp.write_text(
        json.dumps(registro, indent=2),
        encoding="utf-8"
    )
    os.replace(tmp, LANCAMENTO)

    print("AVALIAÇÃO DINO 672 LANÇADA")
    print("PID:", processo.pid)
    print("Log:", LOG)
    print("Resultados:", SAIDA)
    print("Lançamento solicitado; execução ainda não verificada.")

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import json
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

DINO_256 = (
    ROOT / "reproducao/baseline_dino_mri_256_fold0_alp"
)
DINO_672 = (
    ROOT / "reproducao/baseline_dino_mri_672_fold0_alp"
)

ORGAOS = ["RK", "LK"]
PACIENTES = [0, 1, 2, 3]
TOTAL = 112

def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

def ler_csv(path):
    if not path.is_file():
        return []

    registros = []
    with path.open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            try:
                # Ignorar eventual linha incompleta.
                float(r["dice"])
                int(r["paciente"])
                int(r["fatia"])
                registros.append(r)
            except (ValueError, TypeError, KeyError):
                continue

    return registros

def chave(r):
    return (
        r["orgao"],
        int(r["paciente"]),
        int(r["fatia"])
    )

# ==========================================================
# 1. Processo
# ==========================================================

lancamento = ler_json(DINO_672 / "lancamento.json")
status = ler_json(DINO_672 / "status.json")

pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmd = (proc / "cmdline").read_bytes()
        estado = (proc / "stat").read_text().split(") ")[1][0]

        ativo = (
            b"avaliar_dino_alp_mri_672.py" in cmd
            and estado not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

registros_256 = ler_csv(
    DINO_256 / "metricas_fatias.csv"
)
registros_672 = ler_csv(
    DINO_672 / "metricas_fatias.csv"
)

mapa_256 = {chave(r): r for r in registros_256}
mapa_672 = {chave(r): r for r in registros_672}

n = len(registros_672)
progresso = min(100, n / TOTAL * 100)

print("=" * 68)
print("CHAOS MRI — DINOv2-L + ALP: 256 versus 672")
print("=" * 68)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))
print(f"Inferências: {n}/{TOTAL}")
print(f"Progresso: {progresso:.2f}%")

blocos = min(30, int(progresso * 30 / 100))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

# ==========================================================
# 2. Verificar os casos
# ==========================================================

contagem = Counter(r["orgao"] for r in registros_672)

print("\n=== PROGRESSO POR ÓRGÃO ===")
print(f"Rim direito: {contagem['RK']}/55")
print(f"Rim esquerdo: {contagem['LK']}/57")

integridade = (
    len(registros_256) == 112
    and len(mapa_256) == 112
    and len(mapa_672) == n
    and set(mapa_672).issubset(set(mapa_256))
)

completo = (
    integridade
    and n == TOTAL
    and set(mapa_672) == set(mapa_256)
)

print("\n=== INTEGRIDADE ===")
print("Casos válidos, sem duplicações:", integridade)
print("Mesmas 112 fatias nos dois experimentos:", completo)

# ==========================================================
# 3. Resultados completos
# ==========================================================

if completo:
    for k in mapa_256:
        assert int(mapa_256[k]["pixels_gt"]) == int(
            mapa_672[k]["pixels_gt"]
        ), f"Ground truth divergente: {k}"

    print("\n=== DICE MÉDIO ± DP ENTRE PACIENTES ===")
    print()
    print(
        f"{'Órgão':<16}"
        f"{'DINO 256':>19}"
        f"{'DINO 672':>20}"
        f"{'Δ p.p.':>10}"
    )
    print("-" * 67)

    medias_256_geral = defaultdict(list)
    medias_672_geral = defaultdict(list)
    deltas_pacientes = defaultdict(list)

    for orgao in ORGAOS:
        valores_256 = []
        valores_672 = []

        for pid_query in PACIENTES:
            chaves = sorted(
                k for k in mapa_256
                if k[0] == orgao and k[1] == pid_query
            )

            assert chaves

            d256 = float(np.mean([
                float(mapa_256[k]["dice"])
                for k in chaves
            ]))

            d672 = float(np.mean([
                float(mapa_672[k]["dice"])
                for k in chaves
            ]))

            valores_256.append(d256)
            valores_672.append(d672)

            medias_256_geral[pid_query].append(d256)
            medias_672_geral[pid_query].append(d672)
            deltas_pacientes[pid_query].append(d672 - d256)

        media_256 = float(np.mean(valores_256))
        media_672 = float(np.mean(valores_672))

        dp_256 = float(np.std(valores_256, ddof=1))
        dp_672 = float(np.std(valores_672, ddof=1))

        nome = (
            "Rim direito" if orgao == "RK"
            else "Rim esquerdo"
        )

        print(
            f"{nome:<16}"
            f"{media_256:8.4f} ± {dp_256:.4f}"
            f"{media_672:9.4f} ± {dp_672:.4f}"
            f"{(media_672-media_256)*100:+10.2f}"
        )

    # Cada paciente contribui com a média dos dois rins.
    geral_256 = [
        float(np.mean(medias_256_geral[pid]))
        for pid in PACIENTES
    ]
    geral_672 = [
        float(np.mean(medias_672_geral[pid]))
        for pid in PACIENTES
    ]

    delta_geral = np.array(geral_672) - np.array(geral_256)

    print("-" * 67)
    print(
        f"{'Média rins':<16}"
        f"{np.mean(geral_256):12.4f}"
        f"{np.mean(geral_672):20.4f}"
        f"{np.mean(delta_geral)*100:+10.2f}"
    )

    print("\n=== COMPARAÇÃO PAREADA POR PACIENTE ===")
    print("Diferença positiva = vantagem de 672")
    print()
    print(
        f"{'Paciente':>9} | "
        f"{'Δ direito':>11} | "
        f"{'Δ esquerdo':>12} | "
        f"{'Δ médio':>10}"
    )
    print("-" * 54)

    for pid in PACIENTES:
        dr, dl = deltas_pacientes[pid]

        print(
            f"{pid:9d} | "
            f"{dr*100:+10.2f} | "
            f"{dl*100:+11.2f} | "
            f"{(dr+dl)*50:+9.2f}"
        )

    print("\n=== RESUMO DA DIFERENÇA ===")

    print(
        "Ganho médio 672 versus 256:",
        f"{np.mean(delta_geral)*100:+.2f} p.p."
    )
    print(
        "DP das diferenças entre pacientes:",
        f"{np.std(delta_geral, ddof=1)*100:.2f} p.p."
    )
    print(
        "Pacientes que melhoraram:",
        f"{sum(x > 0 for x in delta_geral)}/4"
    )

    print("\n=== PREVISÕES VAZIAS ===")

    for nome, mapa in [
        ("DINO 256", mapa_256),
        ("DINO 672", mapa_672)
    ]:
        vazias = sum(
            int(r["pixels_pred"]) == 0
            for r in mapa.values()
        )
        print(f"{nome}: {vazias}/112")

else:
    print("\nAguardando os 112 casos para calcular as médias.")

# ==========================================================
# 4. Diagnóstico e log
# ==========================================================

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "Sem detalhes."))

elif status.get("estado") == "concluido" and completo:
    print("AVALIAÇÃO DINO 672 + ALP CONCLUÍDA")

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

log = DINO_672 / "console.log"

if log.is_file():
    with log.open(
        encoding="utf-8", errors="replace"
    ) as f:
        for linha in deque(f, maxlen=18):
            print(linha.rstrip())
else:
    print("Log ainda não disponível.")

In [ ]:
from pathlib import Path
import importlib.util
import csv
import json
import gc

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = (
    ROOT / "exp/baseline_dino_mri_v1"
    / "avaliar_dino_alp_sam_mri.py"
)

REF_ALP = (
    ROOT / "reproducao/baseline_dino_mri_672_fold0_alp"
    / "metricas_fatias.csv"
)

assert SCRIPT.is_file()
assert REF_ALP.is_file()
assert torch.cuda.is_available()

spec = importlib.util.spec_from_file_location(
    "baseline_dino_sam_256_referencia", SCRIPT
)

mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False

print("=" * 64)
print("DIAGNÓSTICO — DINOv2-L + ALP + SAM-H, 672")
print("=" * 64)

modelo = None
protosam = None

try:
    # 1. Construir encoder congelado em 672
    modelo = mod.FewShotSegDinoLocal(
        image_size=672,
        pretrained_path=None,
        cfg=mod.CONFIG_ALP.copy()
    ).to("cuda", dtype=torch.float32)

    modelo.eval().requires_grad_(False)

    assert modelo.config["feature_hw"] == [48, 48]
    assert not any(p.requires_grad for p in modelo.parameters())

    # 2. Construir SAM com a mesma configuração validada.
    cfg = dict(
        json.loads(mod.CFG_SAM.read_text())["sam"]
    )

    cfg["image_size"] = [1024, 1024]
    cfg["alpnet_image_size"] = 672
    cfg["sam_pretrained_path"] = str(
        ROOT / "pretrained_model/sam_vit_h.pth"
    )

    assert cfg["use_bbox"] and cfg["use_points"]
    assert not cfg["use_mask"]

    print("Entrada DINO:", 672)
    print("Grade ALP:", modelo.config["feature_hw"])
    print("Canvas SAM:", cfg["image_size"])
    print("Encoder congelado: SIM")

    protosam = mod.ProtoSAM(
        coarse_segmentation_model=mod.ALPNetWrapper(modelo),
        **cfg
    ).to("cuda", dtype=torch.float32)

    protosam.eval().requires_grad_(False)

    # 3. Recuperar exatamente o mesmo suporte/query.
    classe = 2  # Rim direito

    imagens_s, labels_s = mod.volume(4)
    imagens_q, labels_q = mod.volume(0)

    positivas_s = np.flatnonzero(
        np.any(labels_s == classe, axis=(1, 2))
    )
    positivas_q = np.flatnonzero(
        np.any(labels_q == classe, axis=(1, 2))
    )

    zs = int(positivas_s[len(positivas_s) // 2])
    zq = int(positivas_q[len(positivas_q) // 2])

    support_256 = mod.preparar(imagens_s[zs])
    query_256 = mod.preparar(imagens_q[zq])

    mask_256 = torch.from_numpy(
        (labels_s[zs] == classe).astype(np.float32)
    )[None, None].to("cuda")

    support = F.interpolate(
        support_256,
        size=(672, 672),
        mode="bilinear",
        align_corners=False
    )

    query = F.interpolate(
        query_256,
        size=(672, 672),
        mode="bilinear",
        align_corners=False
    )

    mask = F.interpolate(
        mask_256,
        size=(672, 672),
        mode="nearest"
    )[:, 0]

    gt = labels_q[zq] == classe

    print(f"\nSuporte: paciente 4, fatia {zs}")
    print(f"Query: paciente 0, fatia {zq}")

    # 4. Reproduzir a inferência ALP já salva.
    with torch.inference_mode():
        retorno = modelo(
            [[support]],
            [[mask]],
            [[1 - mask]],
            [query],
            isval=True,
            val_wsize=2,
            show_viz=False
        )

        logits = retorno[0]

        assert logits.shape == (1, 2, 672, 672)

        logits_256 = F.interpolate(
            logits,
            size=(256, 256),
            mode="bilinear",
            align_corners=False
        )

        alp_pred = (
            logits_256.argmax(dim=1)[0]
            .cpu().numpy() == 1
        )

    def metricas(pred, gt):
        tp = int((pred & gt).sum())
        fp = int((pred & ~gt).sum())
        fn = int((~pred & gt).sum())

        return (
            2 * tp / max(2 * tp + fp + fn, 1),
            tp / max(tp + fp + fn, 1)
        )

    dice_alp, iou_alp = metricas(alp_pred, gt)

    with REF_ALP.open(newline="", encoding="utf-8") as f:
        referencia = [
            r for r in csv.DictReader(f)
            if r["orgao"] == "RK"
            and int(r["paciente"]) == 0
            and int(r["fatia"]) == zq
        ]

    assert len(referencia) == 1

    dice_salvo = float(referencia[0]["dice"])

    print("\n=== VERIFICAÇÃO ALP ===")
    print(f"Dice ALP salvo: {dice_salvo:.6f}")
    print(f"Dice ALP reproduzido: {dice_alp:.6f}")

    assert np.isclose(
        dice_alp, dice_salvo, atol=1e-6
    ), "ALP não reproduziu a avaliação anterior."

    # 5. Executar ALP + SAM-H.
    entrada = mod.ALPNetInput(
        support_images=[support],
        support_labels=[mask],
        query_images=query,
        isval=True,
        val_wsize=2,
        show_viz=False,
        supp_fts=None
    )

    with (
        torch.no_grad(),
        torch.autocast("cuda", enabled=False)
    ):
        pred_sam, scores = protosam(
            query_image=query,
            coarse_model_input=entrada,
            degrees_rotate=0
        )

    pred_sam = (
        pred_sam.detach().cpu().numpy().squeeze()
    )

    print("\n=== SAÍDA DO SAM-H ===")
    print("Shape recebido:", pred_sam.shape)
    print("Tipo:", pred_sam.dtype)
    print("Valores únicos:", np.unique(pred_sam)[:15])

    # A máscara deve ser binária antes do redimensionamento.
    assert np.isin(pred_sam, [0, 1]).all()

    if pred_sam.shape == (1024, 1024):
        # Na implementação ProtoSAM, um retorno vazio
        # nessa resolução pode indicar ausência de prompt.
        assert not pred_sam.any(), (
            "Retorno 1024 não vazio: investigar geometria."
        )
        sam_executado = False
        pred_256 = np.zeros((256, 256), dtype=bool)

    elif pred_sam.shape in ((672, 672), (256, 256)):
        sam_executado = True

        if pred_sam.shape == (672, 672):
            pred_tensor = torch.from_numpy(
                pred_sam.astype(np.float32)
            )[None, None]

            pred_256 = (
                F.interpolate(
                    pred_tensor,
                    size=(256, 256),
                    mode="nearest-exact"
                )[0, 0].numpy() > 0.5
            )
        else:
            pred_256 = pred_sam.astype(bool)

    else:
        raise AssertionError(
            f"Formato SAM inesperado: {pred_sam.shape}"
        )

    dice_sam, iou_sam = metricas(pred_256, gt)

    print("SAM executado:", sam_executado)
    print("Máscara avaliada:", pred_256.shape)

    print("\n=== RESULTADO DO MESMO CASO ===")
    print(f"Dice ALP:       {dice_alp:.4f}")
    print(f"Dice ALP + SAM: {dice_sam:.4f}")
    print(f"IoU ALP + SAM: {iou_sam:.4f}")
    print(f"Δ Dice SAM: {(dice_sam-dice_alp)*100:+.2f} p.p.")
    print("Pixels reais:", int(gt.sum()))
    print("Pixels previstos SAM:", int(pred_256.sum()))

    print("\nDIAGNÓSTICO CONCLUÍDO")

finally:
    # Liberar DINO e SAM deste diagnóstico,
    # inclusive se ocorrer uma exceção.
    del protosam, modelo
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import json
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
REPRO = ROOT / "reproducao"

ALP_672 = REPRO / "baseline_dino_mri_672_fold0_alp"
SAM_672 = REPRO / "baseline_dino_mri_672_fold0_alp_sam"
SAM_256 = REPRO / "baseline_dino_mri_256_fold0_alp_sam"

VMAMBA = (
    REPRO / "eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_sam_rins_f0_7ckpts"
)

TOTAL = 112
ORGAOS = ["RK", "LK"]
PACIENTES = [0, 1, 2, 3]

def ler_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}

def ler_csv(path):
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def chave(r):
    return (
        r["orgao"],
        int(r["paciente"]),
        int(r["fatia"])
    )

def media_por_paciente(mapa, orgao, coluna):
    valores = []
    for pid in PACIENTES:
        fatias = [
            float(r[coluna])
            for k, r in mapa.items()
            if k[0] == orgao and k[1] == pid
        ]
        assert fatias, (orgao, pid)
        valores.append(float(np.mean(fatias)))

    return np.array(valores)

def formatar(valores):
    return (
        f"{np.mean(valores):.4f} ± "
        f"{np.std(valores, ddof=1):.4f}"
    )

# ==========================================================
# 1. Processo e estado
# ==========================================================

lancamento = ler_json(SAM_672 / "lancamento.json")
status = ler_json(SAM_672 / "status.json")

pid = lancamento.get("pid")
ativo = False

if pid:
    try:
        proc = Path(f"/proc/{pid}")
        cmd = (proc / "cmdline").read_bytes()
        estado_proc = (
            (proc / "stat").read_text().split(") ")[1][0]
        )
        ativo = (
            b"avaliar_dino_alp_sam_mri_672.py" in cmd
            and estado_proc not in ("Z", "X")
        )
    except (OSError, IndexError):
        pass

# ==========================================================
# 2. Ler resultados
# ==========================================================

registros = []

for r in ler_csv(SAM_672 / "metricas_fatias.csv"):
    try:
        chave(r)
        float(r["dice_alp"])
        float(r["dice_sam"])
        int(r["sam_executado"])
        registros.append(r)
    except (ValueError, TypeError, KeyError):
        continue

referencia_672 = ler_csv(ALP_672 / "metricas_fatias.csv")
referencia_256 = ler_csv(SAM_256 / "metricas_fatias.csv")

mapa_672 = {chave(r): r for r in registros}
mapa_alp = {chave(r): r for r in referencia_672}
mapa_256 = {chave(r): r for r in referencia_256}

n = len(registros)
progresso = min(100.0, n / TOTAL * 100)

print("=" * 70)
print("CHAOS MRI — DINOv2-L + ALP + SAM-H, 672")
print("=" * 70)

print("PID:", pid)
print("Processo ativo:", "SIM" if ativo else "NÃO")
print("Estado:", status.get("estado", "inicializando"))
print(f"Inferências: {n}/{TOTAL}")
print(f"Progresso: {progresso:.2f}%")

blocos = min(30, int(progresso * 30 / 100))
print("[" + "█" * blocos + "░" * (30 - blocos) + "]")

print("\n=== PROGRESSO POR ÓRGÃO ===")

contagem = Counter(r["orgao"] for r in registros)

print(f"Rim direito: {contagem['RK']}/55")
print(f"Rim esquerdo: {contagem['LK']}/57")

# ==========================================================
# 3. Verificações de integridade
# ==========================================================

integridade = (
    len(mapa_672) == n
    and len(mapa_alp) == TOTAL
    and len(mapa_256) == TOTAL
    and set(mapa_672).issubset(set(mapa_alp))
    and set(mapa_alp) == set(mapa_256)
)

completo = (
    integridade
    and n == TOTAL
    and set(mapa_672) == set(mapa_alp)
)

print("\n=== INTEGRIDADE ===")
print("Chaves válidas, sem duplicações:", integridade)
print("Mesmas 112 fatias:", completo)

if integridade:
    divergencias = []

    for k, r in mapa_672.items():
        if not np.isclose(
            float(r["dice_alp"]),
            float(mapa_alp[k]["dice"]),
            rtol=0,
            atol=1e-8
        ):
            divergencias.append(k)

    print(
        "Divergências com ALP 672 anterior:",
        len(divergencias)
    )

    if divergencias:
        print("Primeiros casos:", divergencias[:5])

# ==========================================================
# 4. Resultados completos
# ==========================================================

if completo and not divergencias:

    print("\n=== DICE MÉDIO ± DP ENTRE PACIENTES ===")
    print()
    print(
        f"{'Órgão':<14} | "
        f"{'ALP 672':>17} | "
        f"{'SAM 672':>17} | "
        f"{'SAM 256':>17}"
    )
    print("-" * 77)

    resumo = {}

    for orgao in ORGAOS:
        alp672 = media_por_paciente(
            mapa_672, orgao, "dice_alp"
        )
        sam672 = media_por_paciente(
            mapa_672, orgao, "dice_sam"
        )
        sam256 = media_por_paciente(
            mapa_256, orgao, "dice_sam"
        )

        resumo[orgao] = {
            "alp672": alp672,
            "sam672": sam672,
            "sam256": sam256
        }

        nome = (
            "Rim direito" if orgao == "RK"
            else "Rim esquerdo"
        )

        print(
            f"{nome:<14} | "
            f"{formatar(alp672):>17} | "
            f"{formatar(sam672):>17} | "
            f"{formatar(sam256):>17}"
        )

    print("\n=== MÉDIA DOS DOIS RINS ===")

    def media_rins(campo):
        return np.mean(
            np.stack([
                resumo[o][campo] for o in ORGAOS
            ]),
            axis=0
        )

    alp672 = media_rins("alp672")
    sam672 = media_rins("sam672")
    sam256 = media_rins("sam256")

    print(f"ALP 672:       {np.mean(alp672):.4f}")
    print(f"ALP + SAM 672: {np.mean(sam672):.4f}")
    print(f"ALP + SAM 256: {np.mean(sam256):.4f}")

    print("\n=== GANHOS PAREADOS ===")

    ganho_sam = sam672 - alp672
    ganho_resolucao = sam672 - sam256

    print(
        "Ganho SAM em 672:",
        f"{np.mean(ganho_sam)*100:+.2f} p.p."
    )
    print(
        "Ganho 672 vs 256 após SAM:",
        f"{np.mean(ganho_resolucao)*100:+.2f} p.p."
    )
    print(
        "DP do ganho 672 vs 256:",
        f"{np.std(ganho_resolucao, ddof=1)*100:.2f} p.p."
    )
    print(
        "Pacientes que melhoraram em 672:",
        f"{sum(ganho_resolucao > 0)}/4"
    )

    print("\n=== CASOS SEM FOREGROUND / PROMPTS ===")

    vazias_alp = {
        k for k, r in mapa_alp.items()
        if int(r["pixels_pred"]) == 0
    }

    sem_sam = {
        k for k, r in mapa_672.items()
        if int(r["sam_executado"]) == 0
    }

    print("ALP 672 sem foreground:", len(vazias_alp))
    print("SAM não executado:", len(sem_sam))
    print(
        "Casos coincidentes:",
        len(vazias_alp & sem_sam)
    )

    # Comparar também com VMamba-Tiny no mesmo protocolo
    vmamba = ler_csv(VMAMBA / "metricas_fatias.csv")

    print("\n=== COMPARAÇÃO ENTRE ENCODERS ===")
    print(f"{'Pipeline':<32}{'Dice médio':>12}")
    print("-" * 46)

    print(
        f"{'DINOv2-L 672 + SAM':<32}"
        f"{np.mean(sam672):12.4f}"
    )
    print(
        f"{'DINOv2-L 256 + SAM':<32}"
        f"{np.mean(sam256):12.4f}"
    )

    for passo in (0, 5000, 10000):
        subset = {
            chave(r): r
            for r in vmamba
            if int(r["checkpoint"]) == passo
        }

        assert len(subset) == TOTAL
        assert set(subset) == set(mapa_672)

        por_orgao = [
            media_por_paciente(
                subset, orgao, "dice_sam"
            ).mean()
            for orgao in ORGAOS
        ]

        print(
            f"{'VMamba-Tiny ' + str(passo) + ' + SAM':<32}"
            f"{np.mean(por_orgao):12.4f}"
        )

else:
    print(
        "\nResultados comparativos completos "
        "ainda não disponíveis."
    )

# ==========================================================
# 5. Diagnóstico e log
# ==========================================================

print("\n=== DIAGNÓSTICO ===")

if status.get("estado") == "erro":
    print("ERRO NA AVALIAÇÃO")
    print(status.get("traceback", "Sem detalhes."))

elif status.get("estado") == "concluido" and completo:
    print("AVALIAÇÃO DINO 672 + SAM-H CONCLUÍDA")

elif ativo:
    print("AVALIAÇÃO EM EXECUÇÃO")

else:
    print("Processo inativo sem conclusão confirmada.")

print("\n=== ÚLTIMAS MENSAGENS DO LOG ===")

log = SAM_672 / "console.log"

if log.is_file():
    with log.open(
        encoding="utf-8",
        errors="replace"
    ) as f:
        for linha in deque(f, maxlen=18):
            print(linha.rstrip())
else:
    print("Log ainda não disponível.")